# 01 — Estrazione dati IvyDB US (OptionMetrics) + CRSP da WRDS
Progetto: Conditional Dispersion — versione US (indice: S&P 500 / SPX)

Cosa scarica (tutto in parquet, una cartella per tabella, un file per anno):
1. Membership storica S&P 500 (CRSP) + link CRSP permno -> OptionMetrics secid
2. Pesi mensili approssimati (market cap CRSP dei membri)
3. Opzioni standardizzate ATM-forward 30/60/91 giorni (stdopd): indice + membri -> per IC e VP
4. Prezzi dei sottostanti (secprd): indice + membri -> per RC, vol realizzata, delta hedge
5. Opzioni listate per il backtest (opprcd): ogni mese lo straddle ~3M ATM su SPX e sui top-50, con storia giornaliera e bid/ask veri
6. Curva zero (zerocd), dividendi (distrd), dividend yield indice (idxdvd), VIX

Prima di lanciare tutto: esegui le celle 1-3 (connessione, controllo tabelle, controllo colonne).
Se un nome di tabella o colonna non torna, si corregge lì prima del pull pesante.

In [ ]:
# Su Colab o WRDS JupyterHub: installa il pacchetto (su JupyterHub di solito è già presente)
# !pip install wrds pyarrow

import os
import pandas as pd
import wrds

START_YEAR = 1996
END_YEAR = 2025          # IvyDB US su WRDS arriva probabilmente ad agosto 2025: lo verifichiamo sotto
OUT = "data_raw"         # su Colab: monta Drive e usa ad es. "/content/drive/MyDrive/dispersion/data_raw"
os.makedirs(OUT, exist_ok=True)

db = wrds.Connection()   # chiede username WRDS personale + password + conferma Duo la prima volta

## 1. Controllo delle tabelle disponibili
IvyDB US su WRDS ha tabelle annuali (es. opprcd2010) nella libreria `optionm`;
alcune installazioni hanno anche viste uniche in `optionm_all`. Il codice usa quelle annuali se esistono.

In [ ]:
om_tables = set(db.list_tables("optionm"))
print(len(om_tables), "tabelle in optionm")
for base in ["opprcd", "secprd", "stdopd", "vsurfd", "zerocd", "distrd", "idxdvd", "fwdprd", "secnmd", "securd"]:
    yearly = sorted(t for t in om_tables if t.startswith(base) and t[len(base):].isdigit())
    print(f"{base:8s} unica={base in om_tables}  annuali={yearly[:1]}...{yearly[-1:]} ({len(yearly)})")

def tbl(base, year=None):
    """Nome completo della tabella: annuale se esiste, altrimenti quella unica."""
    if year is not None and f"{base}{year}" in om_tables:
        return f"optionm.{base}{year}"
    if base in om_tables:
        return f"optionm.{base}"
    raise ValueError(f"Tabella {base} ({year}) non trovata: controlla la lista sopra")

## 2. Controllo colonne (le query sotto assumono questi nomi)

In [ ]:
for t in ["opprcd", "secprd", "stdopd", "zerocd", "distrd", "idxdvd"]:
    try:
        name = tbl(t, END_YEAR - 1) if t in ("opprcd", "secprd", "stdopd") else tbl(t)
        lib, table = name.split(".")
        cols = db.describe_table(lib, table)["name"].tolist()
        print(name, "->", cols)
    except Exception as e:
        print(t, "ERRORE:", e)

## 3. Secid di SPX e VIX, e ultima data disponibile

In [ ]:
idx = db.raw_sql("""
    SELECT DISTINCT secid, ticker, issuer, effect_date
    FROM optionm.secnmd
    WHERE ticker IN ('SPX', 'VIX', 'OEX', 'XEO')
    ORDER BY ticker, effect_date
""")
print(idx)
SPX = 108105   # secid SPX in IvyDB US (verifica nella tabella sopra)
VIX_SECIDS = idx.loc[idx.ticker == "VIX", "secid"].unique().tolist()

last = db.raw_sql(f"SELECT MAX(date) AS last_date FROM {tbl('secprd', END_YEAR)} WHERE secid = {SPX}")
print("Ultima data IvyDB US:", last.iloc[0, 0])

## 4. Membership S&P 500 (CRSP) e link a OptionMetrics
CRSP ha due formati: legacy (crsp.dsp500list) e CIZ (crsp.dsp500list_v2). Proviamo entrambi.

In [ ]:
try:
    sp = db.raw_sql("SELECT permno, start AS mbr_start, ending AS mbr_end FROM crsp.dsp500list")
except Exception:
    sp = db.raw_sql("SELECT permno, mbrstartdt AS mbr_start, mbrenddt AS mbr_end FROM crsp.dsp500list_v2")
sp["mbr_start"] = pd.to_datetime(sp["mbr_start"])
sp["mbr_end"] = pd.to_datetime(sp["mbr_end"])
sp = sp[sp["mbr_end"] >= f"{START_YEAR}-01-01"]
print(len(sp), "periodi di membership dal", START_YEAR)

# Link CRSP <-> OptionMetrics (tabella WRDS; score 1 = match migliore)
link = db.raw_sql("SELECT secid, permno, sdate, edate, score FROM wrdsapps.opcrsphist")
link = link[link["score"] == 1]
link["sdate"] = pd.to_datetime(link["sdate"])
link["edate"] = pd.to_datetime(link["edate"])

members = sp.merge(link, on="permno", how="left")
# tieni solo le righe dove il periodo del link si sovrappone a quello di membership
members = members[(members["sdate"] <= members["mbr_end"]) & (members["edate"] >= members["mbr_start"])]
members["start"] = members[["mbr_start", "sdate"]].max(axis=1)
members["end"] = members[["mbr_end", "edate"]].min(axis=1)
members = members[["permno", "secid", "start", "end"]].drop_duplicates()
members.to_parquet(f"{OUT}/sp500_membership_secid.parquet")

missing = sp[~sp["permno"].isin(members["permno"])]
print(len(members), "periodi con secid;", missing["permno"].nunique(), "permno senza link (da controllare)")

def secids_in_year(y):
    m = members[(members["start"] <= f"{y}-12-31") & (members["end"] >= f"{y}-01-01")]
    return sorted(m["secid"].astype(int).unique().tolist())

## 5. Pesi mensili approssimati: market cap CRSP a fine mese
Approssimazione dichiarata: S&P usa il market cap flottante; qui usiamo prezzo x azioni in circolazione.

In [ ]:
permnos = ",".join(str(int(p)) for p in sp["permno"].unique())
try:
    mcap = db.raw_sql(f"""
        SELECT permno, date, ABS(prc) * shrout AS mcap
        FROM crsp.msf
        WHERE permno IN ({permnos}) AND date >= '{START_YEAR}-01-01'
    """, date_cols=["date"])
except Exception:
    mcap = db.raw_sql(f"""
        SELECT permno, mthcaldt AS date, ABS(mthprc) * shrout AS mcap
        FROM crsp.msf_v2
        WHERE permno IN ({permnos}) AND mthcaldt >= '{START_YEAR}-01-01'
    """, date_cols=["date"])
mcap.to_parquet(f"{OUT}/crsp_mcap_monthly.parquet")

# Pesi a fine mese sui membri attivi, e top-N per la gamba lunga del trade
TOP_N = 50
mc = mcap.dropna(subset=["mcap"]).copy()
mc["date"] = mc["date"] + pd.offsets.MonthEnd(0)
w = mc.merge(members, on="permno")
w = w[(w["start"] <= w["date"]) & (w["end"] >= w["date"])]
w["weight"] = w["mcap"] / w.groupby("date")["mcap"].transform("sum")
w = w.sort_values(["date", "weight"], ascending=[True, False])
w[["date", "permno", "secid", "weight"]].to_parquet(f"{OUT}/sp500_weights_monthly.parquet", index=False)
top = w.groupby("date").head(TOP_N)[["date", "secid", "weight"]]
top.to_parquet(f"{OUT}/top{TOP_N}_monthly.parquet", index=False)
print("Copertura media del peso dei top", TOP_N, ":", round(top.groupby("date")["weight"].sum().mean(), 3))

## 6. Pull annuale: opzioni standardizzate, prezzi, opzioni listate
Filtri lato server per non scaricare decine di milioni di righe inutili.

In [ ]:
def save(df, name, y):
    os.makedirs(f"{OUT}/{name}", exist_ok=True)
    df.to_parquet(f"{OUT}/{name}/{name}_{y}.parquet", index=False)

for y in range(START_YEAR, END_YEAR + 1):
    ids = [SPX] + VIX_SECIDS + secids_in_year(y)
    ids_sql = ",".join(str(i) for i in ids)

    # 6a. Opzioni standardizzate ATM-forward (IV a scadenza costante)
    std = db.raw_sql(f"""
        SELECT secid, date, days, cp_flag, forward_price, strike_price, premium,
               impl_volatility, delta, gamma, vega, theta
        FROM {tbl('stdopd', y)}
        WHERE secid IN ({ids_sql}) AND days IN (30, 60, 91)
          AND date BETWEEN '{y}-01-01' AND '{y}-12-31'
    """, date_cols=["date"])
    save(std, "stdopd", y)

    # 6b. Prezzi sottostanti (close, rendimento, fattore di aggiustamento)
    px = db.raw_sql(f"""
        SELECT secid, date, close, return, cfadj, shrout, volume
        FROM {tbl('secprd', y)}
        WHERE secid IN ({ids_sql})
          AND date BETWEEN '{y}-01-01' AND '{y}-12-31'
    """, date_cols=["date"])
    save(px, "secprd", y)

    print(y, f"std={len(std):,}  px={len(px):,}")

## 6bis. Opzioni listate per il backtest, in due passi (per non scaricare centinaia di milioni di righe)
Passo A: a ogni data di ingresso (primo giorno di borsa del mese) scegliamo, per SPX e per i top-N
         del mese precedente, lo straddle con scadenza mensile standard più vicina a 91 giorni e strike più vicino allo spot.
Passo B: scarichiamo la storia giornaliera solo di quei contratti, dall'ingresso alla scadenza.
Si sceglie un contratto OGNI mese, così lo stesso dataset serve sia alla strategia condizionata sia al benchmark incondizionato.
ATTENZIONE: in IvyDB strike_price è moltiplicato per 1000.

In [ ]:
spx_px = pd.concat(pd.read_parquet(f"{OUT}/secprd/secprd_{y}.parquet") for y in range(START_YEAR, END_YEAR + 1))
spx_dates = spx_px.loc[spx_px["secid"] == SPX, "date"].sort_values()
entries = spx_dates.groupby(spx_dates.dt.to_period("M")).min().reset_index(drop=True)

def chosen_contracts(y):
    ent = entries[entries.dt.year == y]
    if ent.empty:
        return pd.DataFrame()
    prev_me = (ent - pd.offsets.MonthEnd(1)).dt.normalize()
    ids = set(top[top["date"].isin(prev_me)]["secid"].astype(int)) | {SPX}
    ids_sql = ",".join(str(i) for i in sorted(ids))
    dates_sql = ",".join(f"'{d:%Y-%m-%d}'" for d in ent)
    q = db.raw_sql(f"""
        SELECT o.secid, o.date, o.exdate, o.cp_flag, o.strike_price / 1000.0 AS strike,
               o.optionid, o.best_bid, o.best_offer, p.close AS spot
        FROM {tbl('opprcd', y)} o
        JOIN {tbl('secprd', y)} p ON o.secid = p.secid AND o.date = p.date
        WHERE o.secid IN ({ids_sql}) AND o.date IN ({dates_sql})
          AND (o.exdate - o.date) BETWEEN 60 AND 130
          AND o.strike_price / 1000.0 BETWEEN 0.85 * p.close AND 1.15 * p.close
          AND o.best_bid > 0 AND o.best_offer > o.best_bid
    """, date_cols=["date", "exdate"])
    if q.empty:
        return q
    # solo scadenze mensili standard (terzo venerdì; prima del 2015 IvyDB può riportare il sabato)
    q = q[q["exdate"].dt.day.between(15, 22) & q["exdate"].dt.dayofweek.isin([4, 5])]
    # tieni solo i titoli che erano nel top-N del mese precedente (più SPX)
    q["prev_me"] = (q["date"] - pd.offsets.MonthEnd(1)).dt.normalize()
    q = q.merge(top.rename(columns={"date": "prev_me"}), on=["prev_me", "secid"], how="left")
    q = q[(q["secid"] == SPX) | q["weight"].notna()]
    # scadenza più vicina a 91 giorni, poi strike più vicino allo spot con sia call sia put quotate
    q["dte_gap"] = ((q["exdate"] - q["date"]).dt.days - 91).abs()
    q = q[q["dte_gap"] == q.groupby(["secid", "date"])["dte_gap"].transform("min")]
    both = q.groupby(["secid", "date", "exdate", "strike"])["cp_flag"].transform("nunique") == 2
    q = q[both]
    q["k_gap"] = (q["strike"] - q["spot"]).abs()
    q = q[q["k_gap"] == q.groupby(["secid", "date"])["k_gap"].transform("min")]
    # se restano più contratti per lo stesso strike/lato (es. root diverse), tieni quello con spread minore
    q["spread"] = q["best_offer"] - q["best_bid"]
    q = q.sort_values("spread").drop_duplicates(["secid", "date", "cp_flag"])
    return q.rename(columns={"date": "entry_date"})[
        ["secid", "entry_date", "exdate", "cp_flag", "strike", "optionid", "spot", "weight"]]

contracts = []
for y in range(START_YEAR, END_YEAR + 1):
    c = chosen_contracts(y)
    contracts.append(c)
    print(y, "contratti scelti:", len(c))
contracts = pd.concat(contracts, ignore_index=True)
contracts.to_parquet(f"{OUT}/contracts_chosen.parquet", index=False)

In [ ]:
# Passo B: storia giornaliera dei contratti scelti (un contratto aperto nell'anno y scade al più nell'anno y+1)
for y in range(START_YEAR, END_YEAR + 1):
    c = contracts[contracts["entry_date"].dt.year == y]
    if c.empty:
        continue
    oid_sql = ",".join(str(int(o)) for o in c["optionid"].unique())
    parts = []
    for yy in (y, y + 1):
        try:
            t = tbl("opprcd", yy)
        except ValueError:
            continue
        parts.append(db.raw_sql(f"""
            SELECT optionid, secid, date, exdate, cp_flag, strike_price / 1000.0 AS strike,
                   best_bid, best_offer, volume, open_interest, impl_volatility,
                   delta, gamma, vega, theta
            FROM {t}
            WHERE optionid IN ({oid_sql})
              AND date BETWEEN '{c['entry_date'].min():%Y-%m-%d}' AND '{c['exdate'].max():%Y-%m-%d}'
        """, date_cols=["date", "exdate"]))
    h = pd.concat(parts, ignore_index=True)
    save(h, "option_paths", y)
    print(y, f"righe storia contratti: {len(h):,}")

## 7. Tassi, dividendi, dividend yield dell'indice

In [ ]:
zc = db.raw_sql(f"SELECT date, days, rate FROM {tbl('zerocd')} WHERE date >= '{START_YEAR}-01-01'",
                date_cols=["date"])
zc.to_parquet(f"{OUT}/zerocd.parquet", index=False)

all_ids = ",".join(str(i) for i in sorted(set(members["secid"].astype(int))))
dv = db.raw_sql(f"""
    SELECT * FROM {tbl('distrd')}
    WHERE secid IN ({all_ids}) AND ex_date >= '{START_YEAR}-01-01'
""")
dv.to_parquet(f"{OUT}/distrd.parquet", index=False)

iy = db.raw_sql(f"SELECT * FROM {tbl('idxdvd')} WHERE secid = {SPX} AND date >= '{START_YEAR}-01-01'")
iy.to_parquet(f"{OUT}/idxdvd_spx.parquet", index=False)

print("Fatto. File in", OUT)